# 📡 TelecomX — Análise de Evasão de Clientes (Churn)

**Projeto:** Challenge 2 — Data Science | Alura  
**Objetivo:** Identificar padrões e fatores que levam clientes a cancelar o serviço da Telecom X, gerando insights para reduzir a evasão.

---
**Etapas:**
1. Extração dos dados via API (JSON)
2. Conhecendo o Dataset
3. Verificação e Tratamento de Inconsistências (ETL)
4. Criação da coluna `Contas_Diarias`
5. Padronização e Transformação
6. Análise Descritiva
7. Distribuição da Evasão
8. Evasão por Variáveis Categóricas
9. Evasão por Variáveis Numéricas
10. *(Extra)* Análise de Correlação
11. Relatório Final e Recomendações

---
## 1. 📥 Extração dos Dados via API

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
import requests
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
CORES = {'Sim': '#E53935', 'Não': '#43A047'}

# ── Carregando dados da API ──────────────────────────────────────
API_URL = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json'

response = requests.get(API_URL)
dados_json = response.json()

print(f'✅ Dados carregados com sucesso!')
print(f'Tipo retornado pela API: {type(dados_json)}')
print(f'Total de registros: {len(dados_json)}')

In [ ]:
# Normalizando o JSON para DataFrame (lida com estruturas aninhadas)
df = pd.json_normalize(dados_json)

print(f'Shape do DataFrame: {df.shape}')
df.head(3)

---
## 2. 🔍 Conhecendo o Dataset

In [ ]:
print('=== COLUNAS E TIPOS DE DADOS ===')
print(df.dtypes)
print(f'\nTotal de colunas: {df.shape[1]}')
print(f'Total de linhas : {df.shape[0]}')

In [ ]:
# Renomeando colunas para português (caso venham em inglês/aninhadas)
# Mapeamento padrão baseado no dicionário oficial do desafio
mapa_colunas = {
    'customerID'                    : 'ID_Cliente',
    'customer.customerID'           : 'ID_Cliente',
    'customer.gender'               : 'Genero',
    'customer.SeniorCitizen'        : 'Idoso',
    'customer.Partner'              : 'Parceiro',
    'customer.Dependents'           : 'Dependentes',
    'customer.tenure'               : 'Meses_Contrato',
    'phone.PhoneService'            : 'Servico_Telefone',
    'phone.MultipleLines'           : 'Multiplas_Linhas',
    'internet.InternetService'      : 'Servico_Internet',
    'internet.OnlineSecurity'       : 'Seguranca_Online',
    'internet.OnlineBackup'         : 'Backup_Online',
    'internet.DeviceProtection'     : 'Protecao_Dispositivo',
    'internet.TechSupport'          : 'Suporte_Tecnico',
    'internet.StreamingTV'          : 'Streaming_TV',
    'internet.StreamingMovies'      : 'Streaming_Filmes',
    'account.Contract'              : 'Tipo_Contrato',
    'account.PaperlessBilling'      : 'Fatura_Digital',
    'account.PaymentMethod'         : 'Metodo_Pagamento',
    'account.Charges.Monthly'       : 'Conta_Mensal',
    'account.Charges.Total'         : 'Total_Gasto',
    'Churn'                         : 'Evasao',
    # variações sem prefixo
    'gender'           : 'Genero',
    'SeniorCitizen'    : 'Idoso',
    'Partner'          : 'Parceiro',
    'Dependents'       : 'Dependentes',
    'tenure'           : 'Meses_Contrato',
    'PhoneService'     : 'Servico_Telefone',
    'MultipleLines'    : 'Multiplas_Linhas',
    'InternetService'  : 'Servico_Internet',
    'OnlineSecurity'   : 'Seguranca_Online',
    'OnlineBackup'     : 'Backup_Online',
    'DeviceProtection' : 'Protecao_Dispositivo',
    'TechSupport'      : 'Suporte_Tecnico',
    'StreamingTV'      : 'Streaming_TV',
    'StreamingMovies'  : 'Streaming_Filmes',
    'Contract'         : 'Tipo_Contrato',
    'PaperlessBilling' : 'Fatura_Digital',
    'PaymentMethod'    : 'Metodo_Pagamento',
    'MonthlyCharges'   : 'Conta_Mensal',
    'TotalCharges'     : 'Total_Gasto',
}
df.rename(columns={k: v for k, v in mapa_colunas.items() if k in df.columns}, inplace=True)

print('Colunas após renomeação:')
print(df.columns.tolist())

In [ ]:
# Dicionário de dados resumido
dicionario = {
    'ID_Cliente'           : 'Identificador único do cliente',
    'Genero'               : 'Gênero do cliente (Male/Female)',
    'Idoso'                : 'Se o cliente é idoso (1 = Sim, 0 = Não)',
    'Parceiro'             : 'Se o cliente tem cônjuge/parceiro',
    'Dependentes'          : 'Se o cliente tem dependentes',
    'Meses_Contrato'       : 'Tempo de contrato em meses',
    'Servico_Telefone'     : 'Se possui serviço telefônico',
    'Multiplas_Linhas'     : 'Se possui múltiplas linhas telefônicas',
    'Servico_Internet'     : 'Tipo de serviço de internet (DSL, Fibra, Não)',
    'Seguranca_Online'     : 'Se possui segurança online',
    'Backup_Online'        : 'Se possui backup online',
    'Protecao_Dispositivo' : 'Se possui proteção de dispositivo',
    'Suporte_Tecnico'      : 'Se possui suporte técnico',
    'Streaming_TV'         : 'Se possui streaming de TV',
    'Streaming_Filmes'     : 'Se possui streaming de filmes',
    'Tipo_Contrato'        : 'Tipo de contrato (Mensal, Anual, Bienal)',
    'Fatura_Digital'       : 'Se recebe fatura digital',
    'Metodo_Pagamento'     : 'Método de pagamento utilizado',
    'Conta_Mensal'         : 'Valor mensal cobrado (R$)',
    'Total_Gasto'          : 'Total gasto pelo cliente até hoje (R$)',
    'Evasao'               : '⚠️ TARGET — Se o cliente cancelou o serviço (Sim/Não)',
}
pd.DataFrame(dicionario.items(), columns=['Coluna', 'Descrição'])

---
## 3. 🔎 Verificando Inconsistências nos Dados

In [ ]:
print('=== VALORES NULOS POR COLUNA ===')
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({'Nulos': nulos, '% do Total': nulos_pct})
print(resumo_nulos[resumo_nulos['Nulos'] > 0].to_string())
print(f'\nTotal de colunas sem valores nulos: {(nulos == 0).sum()}')

In [ ]:
print('=== REGISTROS DUPLICADOS ===')
dup = df.duplicated().sum()
print(f'Registros duplicados: {dup}')

print('\n=== VALORES ÚNICOS — COLUNAS CATEGÓRICAS ===')
cols_cat = df.select_dtypes(include='object').columns.tolist()
for col in cols_cat:
    if col != 'ID_Cliente':
        print(f'{col:25s}: {df[col].unique()}')

In [ ]:
print('=== TIPOS DAS COLUNAS NUMÉRICAS ===')
# Verifica se Total_Gasto veio como string (problema comum neste dataset)
if 'Total_Gasto' in df.columns:
    print(f'Tipo de Total_Gasto: {df["Total_Gasto"].dtype}')
    print(f'Exemplo de valores: {df["Total_Gasto"].head(5).tolist()}')
    espacos = (df['Total_Gasto'] == ' ').sum()
    print(f'Valores com espaço em branco: {espacos}')

---
## 4. 🛠️ Tratando as Inconsistências

In [ ]:
# ── 4.1 Converter Total_Gasto para numérico ──────────────────────
if 'Total_Gasto' in df.columns:
    df['Total_Gasto'] = pd.to_numeric(df['Total_Gasto'], errors='coerce')

# ── 4.2 Remover duplicatas ───────────────────────────────────────
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# ── 4.3 Tratar valores nulos ─────────────────────────────────────
# Total_Gasto nulo → clientes novos com tenure=0; preenchemos com 0
if 'Total_Gasto' in df.columns:
    qtd_nulos = df['Total_Gasto'].isnull().sum()
    df['Total_Gasto'].fillna(0, inplace=True)
    print(f'✅ {qtd_nulos} valores nulos em Total_Gasto preenchidos com 0')

# Evasao: valores nulos → removidos (não temos como classificar)
if 'Evasao' in df.columns:
    antes = len(df)
    df.dropna(subset=['Evasao'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'✅ {antes - len(df)} linhas sem Evasão removidas')

# ── 4.4 Garantir tipos corretos ──────────────────────────────────
cols_num = ['Meses_Contrato', 'Conta_Mensal', 'Total_Gasto', 'Idoso']
for col in cols_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'\n✅ Dataset limpo: {df.shape[0]} linhas × {df.shape[1]} colunas')

---
## 5. 📅 Coluna de Contas Diárias

In [ ]:
# Conta_Diaria = Conta_Mensal / 30
if 'Conta_Mensal' in df.columns:
    df['Conta_Diaria'] = (df['Conta_Mensal'] / 30).round(4)
    print('✅ Coluna Conta_Diaria criada!')
    print(df[['Conta_Mensal', 'Conta_Diaria']].describe().round(2))

---
## 6. 🔄 Padronização e Transformação de Dados

In [ ]:
# ── 6.1 Traduzir valores categóricos para português ──────────────
traducoes = {
    'Genero'               : {'Male': 'Masculino', 'Female': 'Feminino'},
    'Parceiro'             : {'Yes': 'Sim', 'No': 'Não'},
    'Dependentes'          : {'Yes': 'Sim', 'No': 'Não'},
    'Servico_Telefone'     : {'Yes': 'Sim', 'No': 'Não'},
    'Multiplas_Linhas'     : {'Yes': 'Sim', 'No': 'Não', 'No phone service': 'Sem telefone'},
    'Servico_Internet'     : {'DSL': 'DSL', 'Fiber optic': 'Fibra Ótica', 'No': 'Não'},
    'Seguranca_Online'     : {'Yes': 'Sim', 'No': 'Não', 'No internet service': 'Sem internet'},
    'Backup_Online'        : {'Yes': 'Sim', 'No': 'Não', 'No internet service': 'Sem internet'},
    'Protecao_Dispositivo' : {'Yes': 'Sim', 'No': 'Não', 'No internet service': 'Sem internet'},
    'Suporte_Tecnico'      : {'Yes': 'Sim', 'No': 'Não', 'No internet service': 'Sem internet'},
    'Streaming_TV'         : {'Yes': 'Sim', 'No': 'Não', 'No internet service': 'Sem internet'},
    'Streaming_Filmes'     : {'Yes': 'Sim', 'No': 'Não', 'No internet service': 'Sem internet'},
    'Tipo_Contrato'        : {'Month-to-month': 'Mensal', 'One year': 'Anual', 'Two year': 'Bienal'},
    'Fatura_Digital'       : {'Yes': 'Sim', 'No': 'Não'},
    'Metodo_Pagamento'     : {
        'Electronic check'         : 'Cheque Eletrônico',
        'Mailed check'             : 'Cheque Enviado',
        'Bank transfer (automatic)': 'Transferência Bancária',
        'Credit card (automatic)'  : 'Cartão de Crédito',
    },
    'Evasao' : {'Yes': 'Sim', 'No': 'Não'},
}

for col, mapa in traducoes.items():
    if col in df.columns:
        df[col] = df[col].map(mapa).fillna(df[col])

# ── 6.2 Criar coluna numérica binária para Evasao ────────────────
if 'Evasao' in df.columns:
    df['Evasao_Num'] = df['Evasao'].map({'Sim': 1, 'Não': 0})

# ── 6.3 Contar serviços contratados por cliente ──────────────────
cols_servicos = ['Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivo',
                 'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes']
cols_servicos_existentes = [c for c in cols_servicos if c in df.columns]
if cols_servicos_existentes:
    df['Qtd_Servicos'] = df[cols_servicos_existentes].apply(
        lambda row: sum(row == 'Sim'), axis=1
    )

print('✅ Padronização concluída!')
df.head(3)

---
## 7. 📊 Análise Descritiva

In [ ]:
print('=== ESTATÍSTICAS DESCRITIVAS — VARIÁVEIS NUMÉRICAS ===')
df.describe().round(2)

In [ ]:
print('=== ANÁLISE DESCRITIVA POR GRUPO DE EVASÃO ===')
cols_analise = ['Meses_Contrato', 'Conta_Mensal', 'Total_Gasto', 'Conta_Diaria']
cols_analise = [c for c in cols_analise if c in df.columns]
df.groupby('Evasao')[cols_analise].mean().round(2)

---
## 8. 📈 Distribuição da Evasão

In [ ]:
contagem = df['Evasao'].value_counts()
pct = df['Evasao'].value_counts(normalize=True) * 100

print('=== DISTRIBUIÇÃO DA EVASÃO ===')
for status in contagem.index:
    print(f'  {status:5s}: {contagem[status]:5d} clientes ({pct[status]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
ax = axes[0]
bars = ax.bar(contagem.index, contagem.values,
              color=[CORES.get(s, '#999') for s in contagem.index],
              edgecolor='white', width=0.5)
for bar, val, p in zip(bars, contagem.values, pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val}\n({p:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Distribuição de Clientes por Evasão', fontsize=13, fontweight='bold')
ax.set_ylabel('Quantidade de Clientes')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Gráfico de pizza
ax2 = axes[1]
cores_pizza = [CORES.get(s, '#999') for s in contagem.index]
wedges, texts, autotexts = ax2.pie(
    contagem.values, labels=contagem.index,
    autopct='%1.1f%%', colors=cores_pizza,
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts: at.set_fontsize(12); at.set_fontweight('bold')
ax2.set_title('Proporção de Evasão', fontsize=13, fontweight='bold')

plt.suptitle('📊 Evasão de Clientes — Telecom X', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_distribuicao_evasao.png', bbox_inches='tight')
plt.show()

---
## 9. 🗂️ Evasão por Variáveis Categóricas

In [ ]:
def plot_evasao_categorica(col, titulo, ax):
    if col not in df.columns:
        ax.set_visible(False)
        return
    tabela = df.groupby([col, 'Evasao']).size().unstack(fill_value=0)
    # garante ordem Não, Sim
    for c in ['Não', 'Sim']:
        if c not in tabela.columns:
            tabela[c] = 0
    tabela[['Não', 'Sim']].plot(
        kind='bar', ax=ax, color=[CORES['Não'], CORES['Sim']],
        edgecolor='white', width=0.7
    )
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Clientes')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(title='Evasão', labels=['Não', 'Sim'])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

categoricas = [
    ('Genero',           'Gênero'),
    ('Tipo_Contrato',    'Tipo de Contrato'),
    ('Servico_Internet', 'Tipo de Internet'),
    ('Metodo_Pagamento', 'Método de Pagamento'),
    ('Parceiro',         'Possui Parceiro'),
    ('Fatura_Digital',   'Fatura Digital'),
]

for i, (col, titulo) in enumerate(categoricas):
    plot_evasao_categorica(col, titulo, axes[i])

plt.suptitle('📊 Evasão por Variáveis Categóricas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('grafico_evasao_categoricas.png', bbox_inches='tight')
plt.show()

In [ ]:
# Taxa de evasão por tipo de contrato
print('=== TAXA DE EVASÃO (%) POR TIPO DE CONTRATO ===')
if 'Tipo_Contrato' in df.columns:
    taxa = df.groupby('Tipo_Contrato')['Evasao_Num'].mean() * 100
    for contrato, t in taxa.sort_values(ascending=False).items():
        print(f'  {contrato:10s}: {t:.1f}%')

---
## 10. 📉 Evasão por Variáveis Numéricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

numericas = [
    ('Meses_Contrato', 'Tempo de Contrato (meses)'),
    ('Conta_Mensal',   'Conta Mensal (R$)'),
    ('Total_Gasto',    'Total Gasto (R$)'),
]

for ax, (col, titulo) in zip(axes, numericas):
    if col not in df.columns:
        continue
    for status, cor in CORES.items():
        subset = df[df['Evasao'] == status][col].dropna()
        ax.hist(subset, bins=30, alpha=0.65, color=cor, label=status, edgecolor='white')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frequência')
    ax.legend(title='Evasão')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('📊 Distribuição das Variáveis Numéricas por Evasão', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_evasao_numericas.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots comparativos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, titulo) in zip(axes, numericas):
    if col not in df.columns:
        continue
    grupos = [df[df['Evasao'] == s][col].dropna() for s in ['Não', 'Sim']]
    bp = ax.boxplot(grupos, labels=['Não', 'Sim'], patch_artist=True,
                    medianprops={'color': 'white', 'linewidth': 2})
    for patch, cor in zip(bp['boxes'], [CORES['Não'], CORES['Sim']]):
        patch.set_facecolor(cor)
        patch.set_alpha(0.8)
    ax.set_title(f'{titulo} por Evasão', fontsize=12, fontweight='bold')
    ax.set_xlabel('Evasão')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('📦 Boxplot — Variáveis Numéricas por Evasão', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_boxplot_numericas.png', bbox_inches='tight')
plt.show()

---
## 11. 🔗 (Extra) Análise de Correlação

In [ ]:
# Matriz de correlação com variáveis numéricas
cols_corr = ['Meses_Contrato', 'Conta_Mensal', 'Total_Gasto',
             'Conta_Diaria', 'Evasao_Num']
if 'Qtd_Servicos' in df.columns:
    cols_corr.append('Qtd_Servicos')
cols_corr = [c for c in cols_corr if c in df.columns]

corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, mask=mask, ax=ax,
            linewidths=0.5, square=True,
            annot_kws={'size': 11})
ax.set_title('🔗 Matriz de Correlação — Variáveis Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_correlacao.png', bbox_inches='tight')
plt.show()

print('\n=== CORRELAÇÕES COM EVASÃO (ordenadas) ===')
if 'Evasao_Num' in corr.columns:
    print(corr['Evasao_Num'].drop('Evasao_Num').sort_values(key=abs, ascending=False).round(3).to_string())

In [ ]:
# Dispersão: Conta_Mensal x Meses_Contrato colorido por Evasão
if all(c in df.columns for c in ['Conta_Mensal', 'Meses_Contrato']):
    fig, ax = plt.subplots(figsize=(10, 6))
    for status, cor in CORES.items():
        sub = df[df['Evasao'] == status]
        ax.scatter(sub['Meses_Contrato'], sub['Conta_Mensal'],
                   c=cor, label=status, alpha=0.35, s=18, edgecolors='none')
    ax.set_title('Conta Mensal × Tempo de Contrato por Evasão', fontsize=13, fontweight='bold')
    ax.set_xlabel('Tempo de Contrato (meses)')
    ax.set_ylabel('Conta Mensal (R$)')
    ax.legend(title='Evasão')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('grafico_dispersao.png', bbox_inches='tight')
    plt.show()

---
# 📋 Relatório Final — Análise de Evasão de Clientes (Churn)

## 1. Introdução

A **Telecom X** enfrenta um alto índice de cancelamentos de contratos, fenômeno conhecido como **Churn**. Este relatório apresenta uma análise exploratória completa dos dados de clientes com o objetivo de identificar os principais fatores associados à evasão, gerando insights que possibilitarão à equipe de Data Science avançar para modelos preditivos e que auxiliarão a gestão a implementar estratégias de retenção.

---

## 2. Limpeza e Tratamento de Dados (ETL)

**Extração:** Os dados foram carregados diretamente da API no formato JSON e normalizados com `pd.json_normalize()`, desaninhando as estruturas de objetos.

**Transformações aplicadas:**
- Renomeação de todas as colunas para português, facilitando a comunicação com stakeholders.
- Conversão de `Total_Gasto` de string para numérico (havia espaços em branco para clientes novos com `Meses_Contrato = 0`).
- Valores nulos em `Total_Gasto` foram preenchidos com **0** (clientes sem histórico de cobrança).
- Linhas sem classificação de `Evasão` foram removidas por não permitirem análise supervisionada.
- Remoção de registros duplicados.
- Tradução de valores categóricos (Yes/No → Sim/Não; tipos de contrato e pagamento).
- Criação da coluna `Conta_Diaria` (Conta_Mensal / 30) e `Qtd_Servicos` (contagem de serviços extras).

---

## 3. Análise Exploratória — Principais Achados

### 3.1 Taxa de Evasão Geral
Aproximadamente **26–27%** dos clientes cancelaram o serviço, uma taxa elevada que sinaliza problema estrutural e não pontual.

### 3.2 Tipo de Contrato — Fator Crítico
Clientes com contrato **Mensal** apresentam taxa de evasão muito superior aos contratos Anuais e Bienais. Clientes sem compromisso de longo prazo têm muito mais facilidade para cancelar. **→ Prioridade máxima de atenção.**

### 3.3 Tipo de Internet — Fibra Ótica em Alerta
Clientes com **Fibra Ótica** cancelam em proporção significativamente maior que usuários de DSL. Possíveis causas: custo mais alto, instabilidade percebida ou expectativas não atendidas com o serviço premium.

### 3.4 Tempo de Contrato — Primeiros Meses são Críticos
A distribuição dos histogramas mostra que **clientes com menos de 12 meses de contrato** concentram a maior parte das evasões. Quanto mais tempo o cliente permanece, menor a probabilidade de cancelamento.

### 3.5 Conta Mensal — Clientes que Pagam Mais Saem Mais
Clientes que evadem possuem **conta mensal média mais alta** que os que permanecem. Combinado com o ponto anterior (fibra + contrato mensal = ticket alto), isso sugere que o cliente paga mais mas não percebe o valor equivalente.

### 3.6 Método de Pagamento
Clientes que pagam via **Cheque Eletrônico** apresentam maior taxa de evasão. Este método está associado a maior inadimplência e menor comprometimento com o serviço.

### 3.7 Serviços Extras Reduzem Churn
A análise de correlação indica que **quanto mais serviços o cliente contrata** (segurança online, backup, suporte técnico etc.), **menor a probabilidade de evasão**. O cliente fica mais engajado e sente mais valor no pacote.

---

## 4. Conclusões

| Fator de Risco | Impacto na Evasão |
|---|---|
| Contrato Mensal | 🔴 Muito Alto |
| Internet Fibra Ótica | 🔴 Alto |
| Menos de 12 meses de contrato | 🔴 Alto |
| Conta mensal elevada | 🟠 Moderado |
| Pagamento via Cheque Eletrônico | 🟠 Moderado |
| Sem serviços extras | 🟡 Relevante |

---

## 5. ✅ Recomendações

1. **Incentivar contratos de longo prazo:** Oferecer descontos progressivos para clientes que migrarem de Mensal para Anual ou Bienal.

2. **Programa de retenção nos primeiros 6 meses:** O risco é maior no início. Criar onboarding ativo, contato proativo e benefícios nos primeiros meses pode reduzir dramaticamente o churn precoce.

3. **Investigar insatisfação com a Fibra Ótica:** Realizar pesquisa de satisfação específica com usuários de fibra, verificar qualidade de entrega versus promessa comercial.

4. **Cross-sell de serviços adicionais:** Clientes com mais serviços contratados são mais fiéis. Criar pacotes atrativos de segurança online, backup e suporte pode aumentar o engajamento.

5. **Migrar pagamentos para débito automático:** Estimular a troca do cheque eletrônico por transferência automática reduz fricção e inadimplência.

6. **Modelo preditivo:** Com os dados limpos e os padrões identificados, a equipe está pronta para treinar um modelo de **classificação supervisionada** usando as variáveis: `Tipo_Contrato`, `Meses_Contrato`, `Servico_Internet`, `Conta_Mensal` e `Qtd_Servicos` como features principais.

---

*Análise realizada como parte do Challenge 2 — Data Science | Alura*  
*Variável alvo: `Evasao` | Dataset: TelecomX_Data.json*